In [ ]:
import subprocess, sys, os, time
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate bitsandbytes 'trl==0.29.1' 2>&1 | tail -3")
sh(f'{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1')
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')

In [ ]:
# Việc #5 report/322: sinh câu tuần tự, lịch sử là câu chính mô hình sinh ở các bước trước. S1/101 (GPU0) và GRPO-SPICE ck500 (GPU1).
import os, glob, json, shutil, hashlib, threading
W = "/kaggle/working"; TMP = "/kaggle/temp"; os.makedirs(TMP, exist_ok=True)
OUT = f"{W}/out"; os.makedirs(OUT, exist_ok=True)
T0 = time.time(); TOM = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
import torch
assert torch.cuda.device_count() == 2, "DỪNG: cần 2 GPU (T4 x2)"
KIT = os.path.dirname(glob.glob("/kaggle/input/**/gen_tuantu.py", recursive=True)[0])
for f in ("grpo_spice.py", "build_branch_data.py", "gen_tuantu.py"): shutil.copy(f"{KIT}/{f}", f"{W}/{f}")
os.makedirs(f"{W}/harness", exist_ok=True)
for f in glob.glob(f"{KIT}/harness/*.py"): shutil.copy(f, f"{W}/harness/")
assert md5(f"{W}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DỪNG: grpo_spice.py lệch"
AD = f"{KIT}/adapters"; CK = f"{AD}/ck500"
assert md5(f"{AD}/adapter_s1_seed101/adapter_model.safetensors") == "0c60a4418d54eaf27bf1d27b895538b7"
assert md5(f"{CK}/adapter_model.safetensors") == "491fa6677340393f1e4464c08a0cec98"
AC = f"{KIT}/ac"; CU = f"{KIT}/cu"
n_rec = sum(1 for _ in open(f"{AC}/test.jsonl")); n_img = len(glob.glob(f"{AC}/images/*.png"))
ghi("AndroidControl", AC, "· bản ghi", n_rec, "· ảnh", n_img)
assert n_rec == 6958 and n_img == 6958, "DỪNG: thiếu dữ liệu"
MERGED = f"{TMP}/s1_merged"
DATA = ["--recs", f"{AC}/test.jsonl", "--ocr", f"{AC}/ocr.jsonl", "--images", f"{AC}/images", "--no-q4"]
def chay(ten, cmd, gpu, cwd=W):
    log = f"{OUT}/{ten}.log"; t = time.time()
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-T0)/3600:5.2f} h · GPU{gpu} · {ten} · {L[-1][:100] if L else ''}", flush=True)
            if time.time() - T0 > 11.3 * 3600: q.terminate(); ghi("chạm trần giờ —", ten); break
    ghi(ten, "GPU", gpu, "mã thoát", q.returncode, f"· {(time.time()-t)/3600:.2f} h")
    return q.returncode

In [ ]:
# Hoà S1, rồi hai phép kiểm trước lượt thật (lịch sử = câu chuẩn, đúng đường của lượt test cũ):
#  S1 hoà trên 20 bước chạm phải trùng preds_s1_seed101 >= 16/20 (luật các lượt trước)
#  ck500 trên 20 bước chạm phải trùng pred_ck500_test >= 19/20 (cùng đường sinh, chỉ để chắc gen_tuantu không đổi gì khác)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", AD, "--merged", MERGED], cwd=W, capture_output=True, text=True,
                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print((r.stdout + r.stderr)[-1200:]); assert r.returncode == 0, "DỪNG: hoà lỗi"
def kiem(nh, ck, goc, nguong, gpu):
    p = f"{OUT}/kiem_{nh}.jsonl"
    if os.path.exists(p): os.remove(p)
    chay(f"kiem_{nh}", ["python", "gen_tuantu.py", "--bundle", AD, "--merged", MERGED, *ck, *DATA,
                        "--history", "gold", "--n-tap", "20", "--out", p], gpu)
    G = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(goc))}
    M = [json.loads(l) for l in open(p)]
    x = sum(G.get((d["episode_id"], d["step_id"])) == d["pred"] for d in M)
    ghi(f"[kiểm {nh}] {x}/{len(M)} trùng lượt cũ"); assert len(M) == 20 and x >= nguong, f"DỪNG: kiểm {nh} {x}/20"
ts = [threading.Thread(target=kiem, args=a) for a in
      (("s1", [], f"{CU}/preds_s1_seed101.jsonl", 16, 0), ("ck500", ["--ckpt", CK], f"{CU}/pred_ck500_test.jsonl", 19, 1))]
for t in ts: t.start()
for t in ts: t.join()

In [ ]:
# Lượt thật: 6.958 bước mỗi mô hình, hai mô hình song song trên hai GPU (~5,5 h). Nối tiếp được nếu chạy lại.
NH = {"s1": [], "ck500": ["--ckpt", CK]}
def gen(nh, gpu):
    chay(f"gen_{nh}", ["python", "gen_tuantu.py", "--bundle", AD, "--merged", MERGED, *NH[nh], *DATA,
                       "--history", "self", "--out", f"{OUT}/pred_{nh}_tuantu.jsonl"], gpu)
    n = sum(1 for _ in open(f"{OUT}/pred_{nh}_tuantu.jsonl")); ghi(nh, "đủ câu" if n == 6958 else "THIẾU CÂU", n)
ts = [threading.Thread(target=gen, args=(nh, g)) for nh, g in (("s1", 0), ("ck500", 1))]
for t in ts: t.start()
for t in ts: t.join()

In [ ]:
# Chấm bằng UGround CHỈ những bước chạm có câu khác lượt cũ; câu giống hệt dùng lại kết quả cũ (tra theo bước và câu).
CU_RAW = {"s1": f"{CU}/score_s1_seed101_raw.jsonl", "ck500": f"{CU}/score_ck500_test_raw.jsonl"}
TAPK = {(r["episode_id"], r["step_id"]) for r in map(json.loads, open(f"{AC}/test.jsonl"))
        if r["action"].get("action_type") in ("click", "long_press") and "x" in r["action"]}
def moi(nh):
    cu = {(d["episode_id"], d["step_id"]): d.get("sent") for d in map(json.loads, open(CU_RAW[nh]))}
    P = [json.loads(l) for l in open(f"{OUT}/pred_{nh}_tuantu.jsonl")]
    M = [{"episode_id": d["episode_id"], "step_id": d["step_id"], "pred": d["pred"]} for d in P
         if (d["episode_id"], d["step_id"]) in TAPK and d["pred"] != cu.get((d["episode_id"], d["step_id"]))]
    with open(f"{OUT}/preds_moi_{nh}.jsonl", "w") as f:
        for d in M: f.write(json.dumps(d, ensure_ascii=False) + "\n")
    ghi(nh, "bước chạm có câu mới:", len(M), "/", len(TAPK))
    if M:
        chay(f"cham_{nh}", ["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--data-root", AC,
                            "--preds", f"{OUT}/preds_moi_{nh}.jsonl", "--out", f"{OUT}/score_moi_{nh}.json"], 0 if nh == "s1" else 1)
ts = [threading.Thread(target=moi, args=(nh,)) for nh in ("s1", "ck500")]
for t in ts: t.start()
for t in ts: t.join()

In [ ]:
sh(f"cd {W} && zip -qr tuantu_out.zip out && ls -la {W}")
ghi("KẾT THÚC")